# Master Orchestrator (All Phases)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedRafiAhmedGIS/forest-carbon-pipeline/blob/main/notebooks/99_Master_Orchestrator.ipynb)

Runs the entire 6-phase pipeline end-to-end on synthetic data for the 'Hour 1' quickstart.


In [ ]:
# Setup: locate the repo's `src` package (any cwd inside the repo, or fresh clone)
import os
import subprocess
import sys

REPO = "forest-carbon-pipeline"
if os.path.isdir("src"):
    ROOT = os.getcwd()
elif os.path.isdir(os.path.join("..", "src")):
    ROOT = os.path.abspath("..")
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/SyedRafiAhmedGIS/forest-carbon-pipeline.git"],
                   check=False)
    ROOT = os.path.abspath(REPO)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
if not os.path.isdir(os.path.join(ROOT, "src")):
    raise RuntimeError(
        f"Could not locate the pipeline source at {ROOT}. Open this notebook "
        "from a repo checkout, or check your network so the bootstrap can "
        "clone https://github.com/SyedRafiAhmedGIS/forest-carbon-pipeline"
    )

from src.phase1_demarcate import demarcate_forest_pixels  # noqa: E402
QUARANTINE = os.path.join(ROOT, "quarantine")
print(f"Pipeline package ready (ROOT = {ROOT})")


## Full 6-phase execution

One click, synthetic data, zero setup - the Hour-1 proof that the pipeline works end to end.


In [ ]:
# Full 6-phase execution on synthetic data - the Hour-1 quickstart
import os

import numpy as np

from src.phase1_demarcate import demarcate_forest_pixels
from src.phase2_preprocess import classify_spectral_types
from src.phase3_clubbing import clubbing_stratification
from src.phase4_model import generate_field_campaign, model_forest_stock
from src.phase5_carbon import calculate_carbon_output

np.random.seed(42)
nir_band = np.random.uniform(0.2, 0.8, (100, 100))
red_band = np.random.uniform(0.05, 0.3, (100, 100))
savi = ((nir_band - red_band) / (nir_band + red_band + 0.5)) * 1.5

forest_mask, ndvi, input_hash = demarcate_forest_pixels(
    nir_band, red_band, quarantine_dir=QUARANTINE)
type_map, density_map = classify_spectral_types(ndvi, savi, forest_mask)
stratum_map, allocation = clubbing_stratification(type_map, density_map)
plots = generate_field_campaign(ndvi, savi, forest_mask, stratum_map)
rf_model, agb_map, mean_r2, std_r2 = model_forest_stock(plots, ndvi, savi, stratum_map)
carbon_map, metadata = calculate_carbon_output(agb_map, commit_sha="main")

q_errors = 0
if os.path.isdir(QUARANTINE):
    q_errors = len([f for f in os.listdir(QUARANTINE) if f.endswith(".json")])
forest_px = agb_map > 0

print("\n==== 6-phase run summary ====")
print(f"Forest coverage : {forest_mask.mean() * 100:.1f}%")
print(f"Active strata   : {len(allocation)}")
print(f"Plots dropped   : {len(plots)}")
print(f"CV R2           : {mean_r2:.2f} +/- {std_r2:.2f}")
print(f"AGB map         : {agb_map[forest_px].min():.0f}-{agb_map[forest_px].max():.0f} t/ha (mean {agb_map[forest_px].mean():.1f})")
print(f"Carbon stock    : {metadata['total_carbon_tC']} tC = {metadata['total_co2e_tCO2e']} t CO2e")
print(f"Quarantine      : {q_errors} errors")
print(f"Provenance      : commit=main, input_hash={input_hash[:8]}")
print("\nMaster Orchestrator complete - all 6 phases executed cleanly.")
